In [ ]:
from trading_engine.core import (
    read_data, create_model_state, orchestrate_model_backtests,
    orchestrate_model_simulations, orchestrate_portfolio_optimizations,
    orchestrate_portfolio_simulations, orchestrate_portfolio_aggregation
)

import datetime

In [ ]:
# Define your trading model
from typing import Callable

import polars as pl
from polars import LazyFrame

def MomentumModel(
        trade_ticker: str,
        signal_ticker: str,
        momentum_column: str,
        inverse: bool = True,  # True => long when signal < 0, else flat
        threshold: float = 0.0,  # optional dead-band around 0 to avoid noise
) -> Callable[[LazyFrame], LazyFrame]:
    """
    Generate weights for `trade_ticker` from the momentum of `signal_ticker`.

    inverse=True  -> long when momentum(signal_ticker) < -threshold, else 0
    inverse=False -> long when momentum(signal_ticker) >  threshold, else 0

    Output columns: ["date", trade_ticker]
    """

    def run_model(lf: LazyFrame) -> LazyFrame:
        # Pull only the signal series (lazy)
        sig = (
            lf.filter(pl.col("ticker") == signal_ticker)
            .select([
                pl.col("date"),
                pl.col(momentum_column).alias("sig")
            ])
        )

        # Condition: long when (sig < -threshold) if inverse else (sig > threshold)
        cond = (pl.col("sig") < -threshold) if inverse else (pl.col("sig") > threshold)

        # Vectorized mapping to {1.0, 0.0}; treat null as False via fill_null(False)
        weights = sig.select([
            pl.col("date"),
            pl.when(cond.fill_null(False))
            .then(pl.lit(1.0))
            .otherwise(pl.lit(0.0))
            .cast(pl.Float64)
            .alias(trade_ticker)
        ])

        return weights  # LazyFrame

    return run_model

In [ ]:
models_registry = {
    "RXI_TLT_pml_10": {
        "tickers": ["RXI-US", "TLT-US"],  # define the tickers your model looks at
        "columns": ["close_momentum_10"],  # define the model state columns your model needs
        "function": MomentumModel(
            trade_ticker="RXI-US",
            signal_ticker="TLT-US",
            momentum_column="close_momentum_10",
            inverse=False,
        ),
    }
}

In [ ]:
# 1) experiment config
universe = [
  "SPY-US", "SLV-US", "GLD-US", "TLT-US", "USO-US", "UNG-US", "IXJ-US",
  "KXI-US", "JXI-US", "IXG-US", "IXN-US", "RXI-US", "MXI-US", "EXI-US",
  "IXC-US", "IEI-US", "SHY-US", "BIL-US", "JPXN-US", "INDA-US", "MCHI-US",
  "EZU-US", "IBIT-US", "ETHA-US", "VIXY-US"
]
features = ["close_momentum_10"]                   # keys in FEATURES
models   = ["RXI_TLT_pml_10"]                      # keys in your custom `models_registry`
aggregators = ["model_mvo"]                        # keys in AGGREGATORS
optimizers   = ["mean_variance_constrained"]       # keys in OPTIMIZERS
initial_value = 1_000_000
start_date = datetime.date(2021, 1, 1)
end_date = datetime.date(2025, 1, 1)

In [ ]:
# 2) build model state + prices (cached locally)
lf = read_data()
model_state, prices = create_model_state(
    lf=lf,
    features=features,
    start_date=start_date,
    end_date=end_date,
    universe=universe
)

In [ ]:
# 3) run model backtests + simulations
model_insights = orchestrate_model_backtests(
    model_state=model_state,
    models=models,
    universe=universe,
    registry=models_registry  # pass in your custom models registry instead of pulling default prod registry
)

model_simulations = orchestrate_model_simulations(
    prices=prices,
    model_insights=model_insights
)

In [ ]:
# 4) aggregate + optimize portfolio and simulate
aggregated_insights = orchestrate_portfolio_aggregation(
    model_insights=model_insights,
    backtest_results=model_simulations,
    universe=universe,
    aggregators=aggregators,
)

optimizer_insights = orchestrate_portfolio_optimizations(
    prices=prices,
    aggregated_insights=aggregated_insights,
    universe=universe,
    optimizers=optimizers,
)

optimizer_simulations = orchestrate_portfolio_simulations(
    prices=prices,
    portfolio_insights=optimizer_insights,
    initial_value=initial_value,
)

In [ ]:
# 5) visualize one result (example: mean_variance_constrained)
optimizer_simulations["mean_variance_constrained"]["backtest_metrics"]